# 05 — Build and Deploy the Claims FE Databricks App

Deploys the FastAPI + Gradio app under `./app/` as a Databricks App. The app
serves the same `claims_fe` wheel logic as the Model Serving endpoint from
notebook 03, so we can benchmark them head-to-head in notebook 06.

**Prerequisites** (all done in prior notebooks):
- `claims_fe-0.1.0-py3-none-any.whl` uploaded to `/Volumes/fins_genai/classic_ml/claims_fe_wheels/`
- `fins_genai.classic_ml.fe_test_payloads` Delta table populated
- Model Serving endpoint `claims-fe-transformer` already live (for comparison)

**What this notebook does:**
1. Pulls N sample payloads from the Delta table, writes them to `app/samples.json` (baked into the deployed UI)
2. Uploads the `app/` source tree to a Workspace folder
3. Creates or updates the Databricks App named `claims-fe-app`
4. Prints the final app URL

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.5/857.5 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-bf88c9bd-0e19-490f-8932-8e3bcd60b4b6
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.67.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-bf88c9bd-0e19-490f-8932-8e3bcd60b4b6
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!

In [0]:
CATALOG = "fins_genai"
SCHEMA = "classic_ml"
TABLE_NAME = "fe_test_payloads"
APP_NAME = "claims-fe-app"
N_SAMPLES_FOR_UI = 10

## Locate the app source tree

Assumes this notebook is running inside the repo checkout (Workspace Files or
a Databricks Repo). The `app/` folder should sit next to this file.

In [0]:
import os
from pathlib import Path

# In a Databricks .py notebook, __file__ resolves to the notebook path in recent runtimes;
# fall back to cwd for older environments.
try:
    NB_DIR = Path(__file__).parent
except NameError:
    NB_DIR = Path(os.getcwd())

APP_SRC = NB_DIR / "app"

if not APP_SRC.is_dir():
    raise RuntimeError(
        f"App source not found at {APP_SRCL}. "
        f"Expected to find ./app/ next to this notebook. "
        f"If you're running from a different location, set APP_SRC_LOCAL explicitly."
    )
print(f"App source: {APP_SRC}")
print(f"Files: {[p.name for p in sorted(APP_SRC.iterdir()) if p.is_file()]}")

APP_SOURCE_WORKSPACE_PATH = APP_SRC

App source: /Workspace/Users/q.yu@databricks.com/customer_requests/farmers/FE_endpoint/app
Files: ['app.py', 'app.yaml', 'models.py', 'requirements.txt', 'samples.json', 'transformer_loader.py', 'ui.py']


## Fetch N sample payloads for the UI dropdown

Writes `app/samples.json` — a `{payload_id: payload_json}` dict the Gradio UI
loads at startup. No warehouse lookup at runtime; samples are baked in.

In [0]:
import json

samples_rows = (
    spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
    .select("payload_id", "payload_json")
    .orderBy("payload_id")
    .limit(N_SAMPLES_FOR_UI)
    .collect()
)
samples = {r.payload_id: r.payload_json for r in samples_rows}
samples_path = APP_SOURCE_WORKSPACE_PATH / "samples.json"
samples_path.write_text(json.dumps(samples, indent=2))
print(f"Wrote {len(samples)} samples to {samples_path}  ({samples_path.stat().st_size:,} bytes)")

Wrote 10 samples to /Workspace/Users/q.yu@databricks.com/customer_requests/farmers/FE_endpoint/app/samples.json  (169,663 bytes)


## Create or update the app

If the app doesn't exist yet, create it and wait until it's active. Then trigger
a deploy against the uploaded source path.

In [0]:
from databricks.sdk.service.apps import App
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    existing = w.apps.get(APP_NAME)
    print(f"App '{APP_NAME}' exists — state: {existing.app_status.state if existing.app_status else 'unknown'}")
    app_exists = True
except Exception as exc:
    if "does not exist" in str(exc).lower() or "not found" in str(exc).lower() or "404" in str(exc):
        app_exists = False
        print(f"App '{APP_NAME}' does not exist — will create.")
    else:
        raise

if not app_exists:
    print("Creating app (this provisions compute and may take several minutes)...")
    w.apps.create_and_wait(
        app=App(
            name=APP_NAME,
            description="Claims feature-engineering endpoint — FastAPI + Gradio (track 2 prototype).",
        )
    )
    print("App created.")

App 'claims-fe-app' exists — state: ApplicationState.RUNNING


## Deploy the source

This is where the wheel is pip-installed, the app process starts, and the
endpoint becomes reachable. Expect 2–5 minutes.

In [0]:
print(f"Deploying from: {APP_SOURCE_WORKSPACE_PATH}")
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy_and_wait(
    app_name=APP_NAME,
    app_deployment=AppDeployment(source_code_path=str(APP_SOURCE_WORKSPACE_PATH)),
)
print(f"Deployment status: {deployment.status.state if deployment.status else 'unknown'}")
if deployment.status and deployment.status.message:
    print(f"Message: {deployment.status.message}")

Deploying from: /Workspace/Users/q.yu@databricks.com/customer_requests/farmers/FE_endpoint/app
Deployment status: AppDeploymentState.SUCCEEDED
Message: App started successfully


## App URL + status

In [0]:
import time

for _ in range(10):
    app = w.apps.get(APP_NAME)
    state = app.compute_status.state if app.compute_status else None
    print(f"compute state: {state}  |  url: {app.url}")
    if state and "ACTIVE" in str(state):
        break
    time.sleep(5)

print(f"\nApp URL: {app.url}")
print(f"Health check:   curl {app.url}/health")
print(f"UI:             {app.url}/ui")
print(f"Transform API:  POST {app.url}/transform  with {{'payload_json': '...'}}")

compute state: ComputeState.ACTIVE  |  url: https://claims-fe-app-1444828305810485.aws.databricksapps.com

App URL: https://claims-fe-app-1444828305810485.aws.databricksapps.com
Health check:   curl https://claims-fe-app-1444828305810485.aws.databricksapps.com/health
UI:             https://claims-fe-app-1444828305810485.aws.databricksapps.com/ui
Transform API:  POST https://claims-fe-app-1444828305810485.aws.databricksapps.com/transform  with {'payload_json': '...'}
